# Now the main inferencing pipeline

# Provide your dataframe path


In [ ]:
datapath = "day0 (8).csv"

# Run Once

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
!{sys.executable} -m pip install pytorch_forecasting

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 391.5/391.5 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 55.9 MB/s eta 0:00:00


In [ ]:
loaded_tft = TemporalFusionTransformer.load_from_checkpoint('tft-epoch=03-step=47552.ckpt', weights_only=False)
print("Temporal Fusion Transformer model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


Temporal Fusion Transformer model loaded successfully.


In [ ]:
import tqdm
import pandas as pd
from tensorflow.keras.models import load_model
import numpy as np

# PIPELINE Starts here

In [ ]:
folder = "/content/drive/MyDrive/PCA_Models"

In [ ]:
def join_with_top_padding(df_left, df_right):
    max_len = max(len(df_left), len(df_right))

    def pad(df):
        pad_len = max_len - len(df)
        if pad_len > 0:
            top = pd.DataFrame([[np.nan]*df.shape[1]] * pad_len, columns=df.columns)
            df = pd.concat([top, df], ignore_index=True)
        return df

    df_left  = pad(df_left)
    df_right = pad(df_right)

    return pd.concat([df_left, df_right], axis=1)


In [ ]:
df = pd.read_csv(datapath)
out_df = df['close'].diff()
out_df = pd.concat([out_df, pd.DataFrame([0])], axis = 0, ignore_index = True)

for j in ["PB", "BB", "VB"]:
  for k in range(2, 13):
    col_select = []
    latent_dim = 0
    if(j == "PB"):
      col_select = ["PB"+str(i)+"_T"+str(k) for i in range(1, 19)]
      latent_dim = 4
    elif(j == "BB"):
      col_select = ["BB"+str(i)+"_T"+str(k) for i in range(4, 16)] + ["BB24", "BB25"]
      latent_dim = 3
    elif(j == "VB"):
      col_select = ["VB"+str(i)+"_T"+str(k) for i in range(1, 7)]
      latent_dim = 3

    original_df_temp = df.loc[:, col_select]
    loaded_model = load_model(folder + "/" + j + "/" + j + "_T" + str(k) + "_" + str(latent_dim) + "col.keras")

    df_temp_filtered = original_df_temp.dropna()

    # Create a DataFrame of the correct output shape (latent_dim) and original_df_temp's index, filled with NaN
    final_autoencoder_output = pd.DataFrame(np.nan, index=original_df_temp.index, columns=range(latent_dim))

    if not df_temp_filtered.empty:
        X = df_temp_filtered.values

        # Handle potential division by zero if standard deviation is 0
        std_dev = np.std(X, axis=0)
        mean_val = np.mean(X, axis=0)
        X_scaled = (X - mean_val) / np.where(std_dev == 0, 1, std_dev) # Prevent division by zero

        out = loaded_model.predict(X_scaled, verbose=0)

        # Place the autoencoder output into the final_autoencoder_output DataFrame at the correct indices
        final_autoencoder_output.loc[df_temp_filtered.index, :] = out

    # Rename columns to avoid clashes when concatenating with out_df
    new_columns_prefix = f"{j}_{k}_{latent_dim}_"
    final_autoencoder_output.columns = [new_columns_prefix + str(c) for c in final_autoencoder_output.columns]

    out_df = join_with_top_padding(out_df, final_autoencoder_output)


In [ ]:
out_df.head()

,0,PB_2_4_0,PB_2_4_1,PB_2_4_2,PB_2_4_3,PB_3_4_0,PB_3_4_1,PB_3_4_2,PB_3_4_3,PB_4_4_0,...,VB_9_3_2,VB_10_3_0,VB_10_3_1,VB_10_3_2,VB_11_3_0,VB_11_3_1,VB_11_3_2,VB_12_3_0,VB_12_3_1,VB_12_3_2
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-0.022753,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.009626,0.563222,0.900562,-1.244003,-0.652649,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-0.013127,-0.975907,-0.214803,0.304794,1.001985,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-0.005251,-0.758090,0.112880,-0.197099,0.394617,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
out_df = out_df.iloc[1:-1, :]

In [ ]:
corpus = []
indicator_df = out_df.notna().astype(int)
indicator_df = indicator_df.add_suffix("_not_nan")
result = pd.concat([out_df, indicator_df], axis=1)
result = pd.concat([result, pd.DataFrame(range(len(out_df)))], axis=1)
result = result.fillna(0)
corpus.append(result)

In [ ]:
result

,0,PB_2_4_0,PB_2_4_1,PB_2_4_2,PB_2_4_3,PB_3_4_0,PB_3_4_1,PB_3_4_2,PB_3_4_3,PB_4_4_0,...,VB_10_3_0_not_nan,VB_10_3_1_not_nan,VB_10_3_2_not_nan,VB_11_3_0_not_nan,VB_11_3_1_not_nan,VB_11_3_2_not_nan,VB_12_3_0_not_nan,VB_12_3_1_not_nan,VB_12_3_2_not_nan,0
1,-0.022753,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.009626,0.563222,0.900562,-1.244003,-0.652649,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
3,-0.013127,-0.975907,-0.214803,0.304794,1.001985,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
4,-0.005251,-0.758090,0.112880,-0.197099,0.394617,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
5,0.047256,-0.690938,0.214848,-0.380122,-0.156586,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1552,0.031504,-0.013667,-0.727771,-1.294300,0.159688,0.027365,-1.291769,0.691050,-0.301362,-0.375459,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1552.0
1553,0.001750,-0.549236,-1.941524,0.978418,0.733504,-0.153141,0.374295,0.691931,1.445215,0.484864,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1553.0
1554,0.012252,-0.427683,-1.510431,0.618065,-1.250194,-1.280130,0.725797,0.652174,1.379827,0.475591,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1554.0
1555,0.036755,1.080064,-0.946575,0.486956,-0.232942,-0.209058,0.320147,1.360777,0.468099,0.278325,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0


In [ ]:
if(len(result) < 60):
    padding_df = pd.DataFrame(0, index=np.arange(60 - len(result)), columns=result.columns)
    result = pd.concat([padding_df, result], ignore_index=True)
    #pad result

In [ ]:
cols = list(result.columns)
cols[-1] = "time_idx"
cols[0] = "target"
result.columns = cols
result['group_id'] = 255
# Correctly assign time_idx to be a sequential index from 0 to len(result)-1
result["time_idx"] = np.arange(len(result)).astype(int)
result["target"] = result["target"].astype(float)
result["group_id"] = result["group_id"].astype(str)

In [ ]:
result

,target,PB_2_4_0,PB_2_4_1,PB_2_4_2,PB_2_4_3,PB_3_4_0,PB_3_4_1,PB_3_4_2,PB_3_4_3,PB_4_4_0,...,VB_10_3_1_not_nan,VB_10_3_2_not_nan,VB_11_3_0_not_nan,VB_11_3_1_not_nan,VB_11_3_2_not_nan,VB_12_3_0_not_nan,VB_12_3_1_not_nan,VB_12_3_2_not_nan,time_idx,group_id
1,-0.022753,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,255
2,0.009626,0.563222,0.900562,-1.244003,-0.652649,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,255
3,-0.013127,-0.975907,-0.214803,0.304794,1.001985,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,255
4,-0.005251,-0.758090,0.112880,-0.197099,0.394617,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,255
5,0.047256,-0.690938,0.214848,-0.380122,-0.156586,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,255
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1552,0.031504,-0.013667,-0.727771,-1.294300,0.159688,0.027365,-1.291769,0.691050,-0.301362,-0.375459,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1551,255
1553,0.001750,-0.549236,-1.941524,0.978418,0.733504,-0.153141,0.374295,0.691931,1.445215,0.484864,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1552,255
1554,0.012252,-0.427683,-1.510431,0.618065,-1.250194,-1.280130,0.725797,0.652174,1.379827,0.475591,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1553,255
1555,0.036755,1.080064,-0.946575,0.486956,-0.232942,-0.209058,0.320147,1.360777,0.468099,0.278325,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1554,255


In [ ]:
result = result.iloc[:-1, :]

In [ ]:
result

,target,PB_2_4_0,PB_2_4_1,PB_2_4_2,PB_2_4_3,PB_3_4_0,PB_3_4_1,PB_3_4_2,PB_3_4_3,PB_4_4_0,...,VB_10_3_1_not_nan,VB_10_3_2_not_nan,VB_11_3_0_not_nan,VB_11_3_1_not_nan,VB_11_3_2_not_nan,VB_12_3_0_not_nan,VB_12_3_1_not_nan,VB_12_3_2_not_nan,time_idx,group_id
1,-0.022753,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,255
2,0.009626,0.563222,0.900562,-1.244003,-0.652649,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,255
3,-0.013127,-0.975907,-0.214803,0.304794,1.001985,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,255
4,-0.005251,-0.758090,0.112880,-0.197099,0.394617,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,255
5,0.047256,-0.690938,0.214848,-0.380122,-0.156586,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,255
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1551,-0.019253,0.499380,-0.790767,-0.556138,0.561443,-0.719664,-0.629188,0.904164,-0.205560,0.105886,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1550,255
1552,0.031504,-0.013667,-0.727771,-1.294300,0.159688,0.027365,-1.291769,0.691050,-0.301362,-0.375459,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1551,255
1553,0.001750,-0.549236,-1.941524,0.978418,0.733504,-0.153141,0.374295,0.691931,1.445215,0.484864,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1552,255
1554,0.012252,-0.427683,-1.510431,0.618065,-1.250194,-1.280130,0.725797,0.652174,1.379827,0.475591,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1553,255


In [ ]:
# Rename columns to replace dots with underscores as dots are not allowed in PyTorch Forecasting
result.columns = result.columns.str.replace(".", "_", regex=False)

# Re-define features with the new column names
features = [col for col in result.columns if col not in ["target", "time_idx", "group_id"]]

max_prediction_length = 20
max_encoder_length = 60
#training_cutoff = result["time_idx"].max() - max_prediction_length

inferencer = TimeSeriesDataSet(
    result,
    time_idx="time_idx",
    target="target",
    group_ids=["group_id"],
    min_encoder_length=30,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["group_id"],
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_reals=features,1
    target_normalizer=GroupNormalizer(
        groups=["group_id"], transformation='softplus'
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

batch_size = 60
test_dataloader = inferencer.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

print("Training and validation datasets created successfully.")

Training and validation datasets created successfully.


In [ ]:
len(test_dataloader)

27

In [ ]:
# To correctly inverse transform batch-wise, we need to return 'x' (the input features)
# along with the raw predictions. This will give us the group_ids and target_scale for each batch.
res = loaded_tft.predict(test_dataloader, mode='prediction', return_y=True, return_x=False)

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


In [ ]:
res.y[0][-1][:5]

tensor([ 0.0175, -0.0047, -0.0165, -0.0085, -0.0018], device='cuda:0')